# 🚀 Allan QLoRA — Шаг 1: Установка и настройка

**Запускается один раз** перед началом дообучения.

Что делает этот ноутбук:
- Проверяет GPU и ресурсы Colab
- Устанавливает все нужные библиотеки
- Монтирует Google Drive
- Создаёт структуру папок на Drive
- Инициализирует `progress.json` для отслеживания прогресса
- Скачивает базовую модель и кэширует на Drive (опционально)

---
**Минимальные требования:** Google Colab + GPU T4 (бесплатный тариф), 80 ГБ на Google Drive

## Шаг 1: Проверка GPU и ресурсов

In [ ]:
import subprocess
import os

# Проверка GPU
print("=" * 60)
print("GPU INFO:")
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                             '--format=csv,noheader'], capture_output=True, text=True)
    print(result.stdout)
except Exception as e:
    print(f"nvidia-smi недоступен: {e}")

# Проверка оперативной памяти
print("\nRAM INFO:")
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(result.stdout)

# Проверка диска /content
print("DISK INFO (/content):")
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print(result.stdout)

# Проверяем тип среды
try:
    import google.colab
    IN_COLAB = True
    print("\n✅ Среда: Google Colab")
except ImportError:
    IN_COLAB = False
    print("\n⚠️  Среда: НЕ Google Colab (некоторые функции могут не работать)")

# Проверка CUDA
import torch
print(f"\nPyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")
else:
    print("❌ GPU не найден. Убедитесь что выбрали Runtime → Change runtime type → T4 GPU")

## Шаг 2: Установка библиотек

Установка занимает ~3-5 минут. Будет выполнена один раз.

In [ ]:
print("📦 Установка библиотек для QLoRA...")
print("(это займёт несколько минут)\n")

# Основные пакеты для QLoRA
!pip install -q --upgrade pip
!pip install -q \
    transformers==4.44.2 \
    peft==0.12.0 \
    trl==0.11.1 \
    bitsandbytes==0.43.3 \
    accelerate==0.33.0 \
    datasets==3.0.1 \
    sentencepiece \
    protobuf \
    scipy \
    einops \
    tensorboard

# Flash Attention 2 (ускорение обучения, опционально)
try:
    !pip install -q flash-attn --no-build-isolation
    print("✅ Flash Attention 2 установлен")
except:
    print("⚠️  Flash Attention 2 не установился (не критично, работаем без него)")

print("\n✅ Установка завершена!")

In [ ]:
# Проверка импортов
print("Проверка импортов...")
import importlib

required = [
    ('transformers', 'transformers'),
    ('peft', 'peft'),
    ('trl', 'trl'),
    ('bitsandbytes', 'bitsandbytes'),
    ('accelerate', 'accelerate'),
    ('datasets', 'datasets'),
]

all_ok = True
for name, pkg in required:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', '?')
        print(f"  ✅ {name} == {ver}")
    except ImportError:
        print(f"  ❌ {name} — НЕ установлен!")
        all_ok = False

if all_ok:
    print("\n✅ Все библиотеки на месте. Можно продолжать.")
else:
    print("\n❌ Некоторые пакеты отсутствуют. Перезапустите ячейку установки.")

## Шаг 3: Монтирование Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# Проверка доступного места на Drive
result = subprocess.run(['df', '-h', '/content/drive/MyDrive'],
                        capture_output=True, text=True)
print("Место на Google Drive:")
print(result.stdout)
print("✅ Google Drive подключён")

## Шаг 4: Структура папок на Google Drive

In [ ]:
import os
import json
from pathlib import Path

# ============================================================
# НАСТРОЙКА: Корневая папка проекта на Google Drive
# Измените по желанию
# ============================================================
DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
# ============================================================

# Структура директорий
dirs = {
    'models':       DRIVE_ROOT / 'models',        # базовые модели (кэш)
    'adapters':     DRIVE_ROOT / 'adapters',      # LoRA-адаптеры после каждого чанка
    'checkpoints':  DRIVE_ROOT / 'checkpoints',   # промежуточные чекпоинты
    'datasets':     DRIVE_ROOT / 'datasets',      # кэш датасетов
    'exports':      DRIVE_ROOT / 'exports',       # финальные GGUF/merged модели
    'logs':         DRIVE_ROOT / 'logs',          # логи обучения
}

print("📁 Создание структуры папок на Google Drive...")
for name, path in dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  ✅ {name}: {path}")

# Сохраняем пути в конфиге для использования в других ноутбуках
config = {
    'drive_root': str(DRIVE_ROOT),
    'dirs': {k: str(v) for k, v in dirs.items()}
}
config_path = DRIVE_ROOT / 'config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f"\n✅ Конфиг сохранён: {config_path}")
print(f"\n📂 Корень проекта: {DRIVE_ROOT}")

## Шаг 5: Инициализация progress.json

Файл `progress.json` хранит, какие чанки каждого датасета уже обучены.  
Ноутбук `02_qlora_ru_train.ipynb` автоматически читает его и берёт **следующий** необученный чанк.

In [ ]:
import json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
progress_path = DRIVE_ROOT / 'progress.json'

# Описание датасетов с разбивкой на чанки
# chunk_size — количество примеров в одном чанке (подбирается под Colab T4)
DATASETS_CONFIG = {
    # Инструкционные датасеты (диалоги, вопрос-ответ)
    "ru_turbo_alpaca": {
        "hf_name": "IlyaGusev/ru_turbo_alpaca",
        "split": "train",
        "subset": None,
        "text_column": None,           # будет собираться из instruction+input+output
        "format": "alpaca",
        "chunk_size": 3000,            # ~3к примеров за сессию
        "description": "Русскоязычный Alpaca (~40k инструкций)",
        "approx_total": 40000,
    },
    "ru_turbo_saiga": {
        "hf_name": "IlyaGusev/ru_turbo_saiga",
        "split": "train",
        "subset": None,
        "text_column": None,           # формат messages
        "format": "saiga",
        "chunk_size": 2000,
        "description": "Русская Saiga — диалоги ChatGPT (~40k)",
        "approx_total": 40000,
    },
    "oasst_ru": {
        "hf_name": "IlyaGusev/oasst_ru",
        "split": "train",
        "subset": None,
        "text_column": "text",
        "format": "text",
        "chunk_size": 2000,
        "description": "OpenAssistant на русском",
        "approx_total": 8000,
    },
    "russian_instructions": {
        "hf_name": "Den4ikAI/russian_instructions",
        "split": "train",
        "subset": None,
        "text_column": None,
        "format": "alpaca",
        "chunk_size": 3000,
        "description": "Russian Instructions (~150k инструкций)",
        "approx_total": 150000,
    },
    # Текстовые корпуса (для общего понимания языка)
    "ru_wikipedia": {
        "hf_name": "wikimedia/wikipedia",
        "split": "train",
        "subset": "20231101.ru",
        "text_column": "text",
        "format": "text",
        "chunk_size": 1000,            # статьи Вики длинные, берём меньше
        "description": "Русская Википедия (~1.5М статей, берём чанками)",
        "approx_total": 1500000,
    },
    "ru_news": {
        "hf_name": "IlyaGusev/gazeta",
        "split": "train",
        "subset": None,
        "text_column": "text",
        "format": "text",
        "chunk_size": 3000,
        "description": "Русские новости Gazeta.ru (~63k статей)",
        "approx_total": 63000,
    },
}

# Создаём или дополняем progress.json
if progress_path.exists():
    with open(progress_path, 'r', encoding='utf-8') as f:
        progress = json.load(f)
    print("📄 Существующий progress.json найден, дополняем новыми датасетами...")
else:
    progress = {}
    print("📄 Создаём новый progress.json...")

# Добавляем только отсутствующие датасеты
for ds_name, ds_cfg in DATASETS_CONFIG.items():
    if ds_name not in progress:
        import math
        total_chunks = math.ceil(ds_cfg['approx_total'] / ds_cfg['chunk_size'])
        progress[ds_name] = {
            'description': ds_cfg['description'],
            'hf_name': ds_cfg['hf_name'],
            'chunk_size': ds_cfg['chunk_size'],
            'approx_total': ds_cfg['approx_total'],
            'total_chunks': total_chunks,
            'completed_chunks': [],      # список завершённых индексов чанков
            'next_chunk': 0,             # следующий чанк для обучения
            'training_sessions': [],     # история сессий обучения
            'adapter_paths': {},         # {chunk_idx: путь к адаптеру}
        }
        print(f"  + Добавлен: {ds_name} ({total_chunks} чанков по {ds_cfg['chunk_size']} примеров)")
    else:
        print(f"  = Уже есть: {ds_name} (чанков завершено: {len(progress[ds_name]['completed_chunks'])}/{progress[ds_name]['total_chunks']})")

# Сохраняем
with open(progress_path, 'w', encoding='utf-8') as f:
    json.dump(progress, f, ensure_ascii=False, indent=2)

print(f"\n✅ progress.json сохранён: {progress_path}")
print("\n📊 Итого датасетов для обучения:")
for ds_name, info in progress.items():
    done = len(info['completed_chunks'])
    total = info['total_chunks']
    print(f"  • {ds_name}: {done}/{total} чанков — {info['description']}")

## Шаг 6: Выбор и предзагрузка базовой модели

Рекомендуемые модели для бесплатного Colab T4 (15 ГБ VRAM):

| Модель | Размер | VRAM с 4-bit | Качество RU |
|--------|--------|--------------|-------------|
| `Qwen/Qwen2.5-3B-Instruct` | 3B | ~3 ГБ | Хорошее |
| `Qwen/Qwen2.5-7B-Instruct` | 7B | ~5-6 ГБ | Отличное |
| `mistralai/Mistral-7B-v0.3` | 7B | ~5-6 ГБ | Отличное |
| `IlyaGusev/saiga_llama3_8b` | 8B | ~6-7 ГБ | Лучшее для RU |

**Рекомендация:** `Qwen/Qwen2.5-7B-Instruct` — лучший баланс для RU на T4.

In [ ]:
# ============================================================
# НАСТРОЙКА: Выберите базовую модель
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # рекомендуется
# BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct" # если T4 перегружен
# BASE_MODEL = "IlyaGusev/saiga_llama3_8b" # лучшее RU качество
# BASE_MODEL = "mistralai/Mistral-7B-v0.3"
# ============================================================

print(f"Выбрана базовая модель: {BASE_MODEL}")

# Сохраняем выбор в конфиг
config_path = DRIVE_ROOT / 'config.json'
with open(config_path, 'r', encoding='utf-8') as f:
    config = json.load(f)

config['base_model'] = BASE_MODEL
config['model_cache_dir'] = str(DRIVE_ROOT / 'models')

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f"✅ Модель записана в конфиг: {config_path}")

In [ ]:
# Опционально: предзагрузить модель на Drive (кэш ~14-15 ГБ)
# Это сэкономит время на каждой следующей сессии Colab.
# ВНИМАНИЕ: потребует ~15 ГБ места на Drive!

PREDOWNLOAD_MODEL = True  # False — пропустить предзагрузку

if PREDOWNLOAD_MODEL:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch
    import json
    from pathlib import Path

    DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')
    with open(DRIVE_ROOT / 'config.json') as f:
        config = json.load(f)

    model_cache = Path(config['model_cache_dir']) / BASE_MODEL.replace('/', '_')

    if model_cache.exists() and any(model_cache.iterdir()):
        print(f"✅ Модель уже закэширована: {model_cache}")
    else:
        print(f"📥 Загрузка токенизатора {BASE_MODEL}...")
        tokenizer = AutoTokenizer.from_pretrained(
            BASE_MODEL,
            cache_dir=str(model_cache),
            trust_remote_code=True
        )
        tokenizer.save_pretrained(str(model_cache / 'tokenizer'))
        print("✅ Токенизатор сохранён")

        print(f"\n📥 Загрузка весов модели {BASE_MODEL}...")
        print("(это займёт несколько минут)")
        # Загружаем только токенизатор и конфиг для проверки
        # Сами веса будут скачаны при первом запуске тренировки
        from transformers import AutoConfig
        model_config = AutoConfig.from_pretrained(
            BASE_MODEL,
            cache_dir=str(model_cache),
            trust_remote_code=True
        )
        print(f"✅ Конфиг модели загружен: {model_config.model_type}")
        print(f"   Параметры: {model_config.num_hidden_layers} слоёв, {model_config.hidden_size} hidden_size")
        print(f"\n⚠️  Сами веса (~14 ГБ) скачаются при первом запуске тренировки (ноутбук 02).")
        print(f"   Место для кэша: {model_cache}")
else:
    print("⏭️  Предзагрузка пропущена. Модель скачается при первом запуске тренировки.")

## Готово!

Настройка завершена. Теперь можно переходить к **`02_qlora_ru_train.ipynb`**.

### Структура проекта на Google Drive:
```
AllanQLoRA/
├── config.json          ← настройки проекта
├── progress.json        ← прогресс обучения по чанкам
├── models/              ← кэш базовых моделей
├── adapters/            ← LoRA-адаптеры (сохраняются после каждого чанка)
├── checkpoints/         ← промежуточные чекпоинты
├── datasets/            ← кэш датасетов
├── exports/             ← финальные модели (GGUF, merged)
└── logs/                ← логи обучения
```

### Порядок работы:
1. ✅ `01_qlora_ru_setup.ipynb` — запущен (этот ноутбук)
2. 🔁 `02_qlora_ru_train.ipynb` — запускать повторно для каждого чанка
3. 🏁 `03_qlora_ru_merge_export.ipynb` — объединить адаптеры и экспортировать в GGUF

In [ ]:
import json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AllanQLoRA')

print("=" * 60)
print("ИТОГОВЫЙ СТАТУС НАСТРОЙКИ")
print("=" * 60)

config_path = DRIVE_ROOT / 'config.json'
progress_path = DRIVE_ROOT / 'progress.json'

with open(config_path) as f:
    config = json.load(f)
with open(progress_path) as f:
    progress = json.load(f)

print(f"\n📁 Проект: {config['drive_root']}")
print(f"🤖 Базовая модель: {config.get('base_model', 'не выбрана')}")
print(f"\n📊 Датасеты ({len(progress)} шт.):")

total_chunks_all = 0
for ds_name, info in progress.items():
    total_chunks_all += info['total_chunks']
    print(f"  • {ds_name}:")
    print(f"    {info['description']}")
    print(f"    Чанков: {info['total_chunks']} × {info['chunk_size']} примеров")

print(f"\n⏱️  Всего чанков для обучения: {total_chunks_all}")
print(f"   (каждый чанк = ~1 сессия Colab)")
print("\n✅ Готово! Запускайте 02_qlora_ru_train.ipynb")